In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [2]:
!wget -q --show-progress \
"https://zenodo.org/records/11101338/files/Fitzpatrick17k_CategorizedAbbrvs.zip?download=1" \
-O /content/Fitzpatrick17k_CategorizedAbbrvs.zip

!unzip -q /content/Fitzpatrick17k_CategorizedAbbrvs.zip -d /content/F17K

/content/Fitzpatric  51%[=========>          ] 695.94M   887KB/s    in 15m 51s 
/content/Fitzpatric 100%[++++++++++=========>]   1.32G  1.09MB/s    in 12m 14s 


In [3]:
from pathlib import Path

images = list(Path("/content/F17K").rglob("*.jpg"))
print("Images found:", len(images))

Images found: 16577


In [4]:
from google.colab import files

uploaded = files.upload()

Saving fitzpatrick17k  raw.csv to fitzpatrick17k  raw.csv
Saving Fitzpatrick17k-C.csv to Fitzpatrick17k-C.csv


In [6]:
import pandas as pd

print(uploaded.keys())

f17k_file = [name for name in uploaded.keys() if "raw" in name.lower()][0]
f17kc_file = [name for name in uploaded.keys() if "17k-c" in name.lower()][0]

f17k = pd.read_csv(f17k_file)
f17kc = pd.read_csv(f17kc_file)

print("F17k:", len(f17k), "rows")
print("F17k classes:", f17k["label"].nunique())

print("\nF17k-C:", len(f17kc), "rows")
print("F17k-C classes:", f17kc["label"].nunique())

print("\nF17k columns:")
print(f17k.columns.tolist())

print("\nF17k-C columns:")
print(f17kc.columns.tolist())

display(f17k.head(3))
display(f17kc.head(3))

dict_keys(['fitzpatrick17k  raw.csv', 'Fitzpatrick17k-C.csv'])
F17k: 16577 rows
F17k classes: 114

F17k-C: 11394 rows
F17k-C classes: 114

F17k columns:
['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']

F17k-C columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name']


,md5hash,fitzpatrick_scale,fitzpatrick_centaur,label,nine_partition_label,three_partition_label,qc,url,url_alphanum
0,5e82a45bc5d78bd24ae9202d194423f8,3,3,drug induced pigmentary changes,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicmminoc...
1,fa2911a9b13b6f8af79cb700937cc14f,1,1,photodermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicpphoto...
2,d2bac3c9e4499032ca8e9b07c7d3bc40,2,3,dermatofibroma,benign dermal,benign,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicdderma...


,filepath,filename,diag,fst,partition,md5hash,fitzpatrick,label,nine_partition_label,three_partition_label,qc,url,url_alphanum,orig_img_name,new_img_name
0,ne-de_361/ne-de_f2_61_3a96728f.jpg,ne-de_f2_61_3a96728f,ne-de,2,train,3a96728fb161fc25b3597d3996dc4e4a,2,neutrophilic dermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppyode...,3a96728fb161fc25b3597d3996dc4e4a.jpg,ne-de_f2_61_3a96728f.jpg
1,lu-er_410/lu-er_f1_89_07be6be3.jpg,lu-er_f1_89_07be6be3,lu-er,1,train,07be6be3cbebd04ff9bfcab50b75378b,1,lupus erythematosus,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicssubac...,07be6be3cbebd04ff9bfcab50b75378b.jpg,lu-er_f1_89_07be6be3.jpg
2,me_261/me_f0_194_72bcd875.jpg,me_f0_194_72bcd875,me,0,train,72bcd875e7a573b4139883059e69328d,-1,melanoma,malignant melanoma,malignant,NaN,http://atlasdermatologico.com.br/img?imageId=4264,httpwwwatlasdermatologicocombrimgimageId4264.jpg,72bcd875e7a573b4139883059e69328d.jpg,me_f0_194_72bcd875.jpg


In [7]:
from pathlib import Path
import numpy as np
import torch

from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms


# ============================================================
# PATH
# ============================================================

IMG_ROOT = Path("/content/F17K/Categorized_AbbrvName")

assert IMG_ROOT.exists(), f"Image folder not found: {IMG_ROOT}"


# ============================================================
# TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# ORIGINAL F17K
# ============================================================

base_dataset = datasets.ImageFolder(
    IMG_ROOT
)

print("Total F17k images:", len(base_dataset))
print("Total classes:", len(base_dataset.classes))


# Same images, different transforms
train_full_dataset = datasets.ImageFolder(
    IMG_ROOT,
    transform=train_transform
)

val_full_dataset = datasets.ImageFolder(
    IMG_ROOT,
    transform=eval_transform
)


# ============================================================
# 90/10 STRATIFIED TRAIN / VALIDATION SPLIT
# ============================================================

indices = np.arange(len(base_dataset))
targets = np.array(base_dataset.targets)

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.10,
    random_state=42,
    stratify=targets
)

train_dataset = Subset(
    train_full_dataset,
    train_indices
)

val_dataset = Subset(
    val_full_dataset,
    val_indices
)


# ============================================================
# F17K-C DATASET
# ============================================================

class F17KCDataset(Dataset):

    def __init__(
        self,
        dataframe,
        root,
        class_to_idx,
        transform=None
    ):

        self.df = dataframe.copy()
        self.root = Path(root)
        self.class_to_idx = class_to_idx
        self.transform = transform

        self.df["filepath"] = (
            self.df["filepath"]
            .astype(str)
            .str.replace("\\", "/", regex=False)
        )

        self.df["class_folder"] = (
            self.df["filepath"]
            .str.split("/")
            .str[0]
        )

        # Check files
        missing = []

        for filepath in self.df["filepath"]:
            path = self.root / filepath

            if not path.exists():
                missing.append(filepath)

        print("F17k-C rows:", len(self.df))
        print("Missing F17k-C images:", len(missing))

        # Check class mapping
        unknown_classes = set(
            self.df["class_folder"]
        ) - set(self.class_to_idx.keys())

        print("Unknown classes:", len(unknown_classes))

        if missing:
            raise RuntimeError(
                f"{len(missing)} F17k-C images are missing."
            )

        if unknown_classes:
            raise RuntimeError(
                f"Unknown F17k-C classes: {unknown_classes}"
            )


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = self.root / row["filepath"]

        image = Image.open(image_path).convert("RGB")

        label = self.class_to_idx[
            row["class_folder"]
        ]

        if self.transform:
            image = self.transform(image)

        return image, label


test_dataset = F17KCDataset(
    dataframe=f17kc,
    root=IMG_ROOT,
    class_to_idx=base_dataset.class_to_idx,
    transform=eval_transform
)


# ============================================================
# DATA LOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# SUMMARY
# ============================================================

print("\n========== DATASET READY ==========")

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("F17k-C test images:", len(test_dataset))
print("Classes:", len(base_dataset.classes))
print("Batch size:", BATCH_SIZE)

Total F17k images: 16577
Total classes: 114
F17k-C rows: 11394
Missing F17k-C images: 0
Unknown classes: 0

========== DATASET READY ==========
Training images: 14919
Validation images: 1658
F17k-C test images: 11394
Classes: 114
Batch size: 32


In [8]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 114

# Pretrained ConvNeXt-Tiny
weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights)

# Replace final classifier for 114 diseases
model.classifier[2] = nn.Linear(
    model.classifier[2].in_features,
    NUM_CLASSES
)

model = model.to(device)

print("Device:", device)
print("Model: ConvNeXt-Tiny")
print("Output classes:", model.classifier[2].out_features)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 117MB/s]


Device: cuda
Model: ConvNeXt-Tiny
Output classes: 114


In [9]:
import numpy as np

# Get labels from the training subset
train_targets = np.array(base_dataset.targets)[train_indices]

# Class counts
class_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES
)

# Balanced class weights
class_weights = len(train_targets) / (
    NUM_CLASSES * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

print("Training setup ready.")
print("Smallest class:", class_counts.min())
print("Largest class:", class_counts.max())

Training setup ready.
Smallest class: 48
Largest class: 588


In [10]:
from tqdm.auto import tqdm

EPOCHS = 10
best_val_acc = 0.0

history = []

scaler = torch.cuda.amp.GradScaler()


for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Train"
    )

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    train_loss /= train_total

    train_acc = (
        100.0 * train_correct / train_total
    )


    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} - Val"
        )

        for images, labels in val_bar:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.cuda.amp.autocast():

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_loss /= val_total

    val_acc = (
        100.0 * val_correct / val_total
    )


    scheduler.step()


    # =========================
    # SAVE BEST
    # =========================

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "/content/best_convnext_tiny.pth"
        )

        saved = " <-- BEST SAVED"

    else:
        saved = ""


    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })


    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.2f}%"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.2f}%"
        f"{saved}\n"
    )


print(
    "Best validation accuracy:",
    f"{best_val_acc:.2f}%"
)

/tmp/ipykernel_1621/2162778803.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/10 - Train:   0%|          | 0/467 [00:00<?, ?it/s]

/tmp/ipykernel_1621/2162778803.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/10 - Val:   0%|          | 0/52 [00:00<?, ?it/s]

/tmp/ipykernel_1621/2162778803.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...


Epoch 9/10 | Train Loss: 1.1717 | Train Acc: 98.26% | Val Loss: 2.5838 | Val Acc: 60.19% <-- BEST SAVED



Epoch 10/10 - Train:   0%|          | 0/467 [00:00<?, ?it/s]

Epoch 10/10 - Val:   0%|          | 0/52 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a354e135f80>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a354e135f80>    
self._shutdown_workers()Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        if w.is_alive():self._shutdown_workers()
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    assert self._parent_pid == os.getpid(), 'can only test a child process'    
AssertionErrorif w.is_alive():: 
can only test a child process
  File "/usr/lib/


Epoch 10/10 | Train Loss: 1.1625 | Train Acc: 98.57% | Val Loss: 2.5746 | Val Acc: 60.25% <-- BEST SAVED

Best validation accuracy: 60.25%


In [11]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("F17k-C test images:", len(test_dataset))

F17k-C test images: 11394


In [12]:
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tqdm.auto import tqdm


# Load best saved model
model.load_state_dict(
    torch.load(
        "/content/best_convnext_tiny.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

all_labels = []
all_preds = []

top5_correct = 0
total = 0


with torch.inference_mode():

    for images, labels in tqdm(
        test_loader,
        desc="Testing on F17k-C"
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)

        preds = outputs.argmax(dim=1)

        # Top-5 predictions
        top5 = outputs.topk(5, dim=1).indices

        top5_correct += (
            top5 == labels.unsqueeze(1)
        ).any(dim=1).sum().item()

        total += labels.size(0)

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )


# Metrics
accuracy = accuracy_score(
    all_labels,
    all_preds
)

macro_precision = precision_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    all_labels,
    all_preds,
    average="weighted",
    zero_division=0
)

top5_accuracy = top5_correct / total


print("\n========== F17K-C TEST ==========")

print(f"Top-1 Accuracy:      {accuracy * 100:.2f}%")
print(f"Top-5 Accuracy:      {top5_accuracy * 100:.2f}%")
print(f"Macro Precision:     {macro_precision:.4f}")
print(f"Macro Recall:        {macro_recall:.4f}")
print(f"Macro F1:            {macro_f1:.4f}")
print(f"Weighted F1:         {weighted_f1:.4f}")

Testing on F17k-C:   0%|          | 0/357 [00:00<?, ?it/s]


========== F17K-C TEST ==========
Top-1 Accuracy:      93.70%
Top-5 Accuracy:      96.96%
Macro Precision:     0.9266
Macro Recall:        0.9464
Macro F1:            0.9355
Weighted F1:         0.9372
